# BARAM 2026 — 시간 이웃 NWP feature 실험 (v9 후보)

v8 실패(α 상향편향, LB −0.009) 후. 모델이 아니라 **미사용 정보**를 추가:
같은 예보 배치(전일 13시 공개, 01~24시) 내 lag/lead/일평균/anomaly/추세 12개 (`wind_lib.add_temporal`).
NWP 타이밍 오차 평활 — 풍력 예측 고전 기법. 상향 편향과 무관한 순수 정보 추가.

- 기준: v7 구성 (α=0, stuck 0.5, 튜닝GBM, 블렌드 0.7, 시드3) raw = 2023 0.6098 / 2024 0.6176
- 판정: 두 폴드 모두 우위 **+ 편향 가드(평균 예측 CF ≤ 38%)** ← v5·v8 교훈

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS","1")
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd
import torch, torch.nn as nn
torch.set_num_threads(1)
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
import wind_lib as W
from submission.ver_8 import temporal_features as T
from official_metric import group_scores

DEV="mps" if torch.backends.mps.is_available() else "cpu"
SEEDS=[15,0,1]; BLEND=0.7; STUCK_TH=0.3; STUCK_W=0.5
MLP_CFG=dict(h=256,depth=3,drop=0.0,lr=0.0015868563457489381,wd=1e-4,bs=256,emb=4)
GBM_PARAMS=dict(objective="mae",n_estimators=2000,learning_rate=0.020651822836313095,
    num_leaves=63,min_child_samples=300,subsample=0.8,subsample_freq=1,colsample_bytree=0.5,
    reg_lambda=0.1,random_state=42,n_jobs=1,verbose=-1)
RAW=os.environ.get("WIND_RAW_DIR",os.path.expanduser("~/Downloads/open"))
GROUPS=(1,2,3)
FR,TGT={},{}
for g in GROUPS:
    df,tgt=W.load_train(g); TGT[g]=tgt
    FR[g]=T.add_temporal(W.add_spatial(W.build(df,g),"train"))
BASE_ALL=[c for c in W.feature_cols(FR[1]) if c not in W.SPATIAL_COLS and c not in T.TEMPORAL_COLS]+["pc_pred_cf"]
V7_FEATS=W.lean_features(BASE_ALL)+W.SPATIAL_COLS
V9_FEATS=V7_FEATS+T.TEMPORAL_COLS
print("v7:",len(V7_FEATS),"v9:",len(V9_FEATS))

def stuck_frac():
    frames={}
    for fn,pre,n,rate in [("scada_vestas_train.csv","vestas",12,3600.0),
                          ("scada_unison_train.csv","unison",5,4200.0)]:
        d=pd.read_csv(f"{RAW}/train/{fn}",encoding="utf-8-sig",parse_dates=["kst_dtm"])
        d["hour"]=d["kst_dtm"].dt.ceil("h")
        agg={}
        for i in range(1,n+1):
            pw=f"{pre}_wtg{i:02d}_power_kw10m"; ws=f"{pre}_wtg{i:02d}_ws"
            h=d.groupby("hour").agg(pw_m=(pw,"mean"),ws_m=(ws,"mean"))
            agg[i]=pd.DataFrame({f"stuck_{i}":((h.ws_m>=4.0)&(h.pw_m<=0.01*rate)).astype(float),
                                 f"rep_{i}":h.pw_m.notna().astype(float)})
        frames[pre]=pd.concat(agg.values(),axis=1)
    def frac(pre,ids):
        f=frames[pre]
        st=f[[f"stuck_{i}" for i in ids]].sum(axis=1); rp=f[[f"rep_{i}" for i in ids]].sum(axis=1)
        return (st/rp).where(rp>=3)
    return {1:frac("vestas",range(1,7)),2:frac("vestas",range(7,13)),3:frac("unison",range(1,6))}
FRAC=stuck_frac()
for g in GROUPS:
    s=FRAC[g].reindex(FR[g].kst_dtm).to_numpy()
    FR[g]=FR[g].assign(stuck_mask=(np.nan_to_num(s,nan=0.0)>=STUCK_TH))
def wvec(fr): return np.where(fr["stuck_mask"],STUCK_W,1.0)

class MLP(nn.Module):
    def __init__(s,nf,ng=3,h=256,depth=3,drop=0.0,emb=4):
        super().__init__(); s.emb=nn.Embedding(ng,emb)
        L=[nn.Linear(nf+emb,h),nn.GELU(),nn.Dropout(drop)]
        for _ in range(depth-1): L+=[nn.Linear(h,h),nn.GELU(),nn.Dropout(drop)]
        L+=[nn.Linear(h,1)]; s.net=nn.Sequential(*L)
    def forward(s,x,g): return s.net(torch.cat([x,s.emb(g)],-1)).squeeze(-1)

def train_one(pool_tr,feats,seed):
    torch.manual_seed(seed); np.random.seed(seed)
    pool_tr=pool_tr.sort_values("kst_dtm")
    mu=pool_tr[feats].mean(); sd=pool_tr[feats].std()+1e-8
    X=((pool_tr[feats]-mu)/sd).to_numpy(np.float32)
    y=pool_tr["cf"].to_numpy(np.float32); gid=pool_tr["gid"].to_numpy(np.int64)
    wt=pool_tr["w"].to_numpy(np.float32)
    n=len(X); cut=int(n*0.85)
    Xt=torch.tensor(X,device=DEV); yt=torch.tensor(y,device=DEV)
    gt=torch.tensor(gid,device=DEV); wtt=torch.tensor(wt,device=DEV)
    net=MLP(len(feats),**{k:MLP_CFG[k] for k in ("h","depth","drop","emb")}).to(DEV)
    opt=torch.optim.AdamW(net.parameters(),lr=MLP_CFG["lr"],weight_decay=MLP_CFG["wd"])
    best=1e9; st=None; pat=0
    for ep in range(120):
        net.train(); perm=np.random.permutation(np.arange(cut))
        for i in range(0,len(perm),MLP_CFG["bs"]):
            b=torch.tensor(perm[i:i+MLP_CFG["bs"]],device=DEV); opt.zero_grad()
            e=(net(Xt[b],gt[b])-yt[b]).abs()
            ((e*wtt[b]).sum()/(wtt[b].sum()+1e-8)).backward(); opt.step()
        net.eval()
        with torch.no_grad():
            e=(net(Xt[cut:],gt[cut:])-yt[cut:]).abs()
            vl=((e*wtt[cut:]).sum()/(wtt[cut:].sum()+1e-8)).item()
        if vl<best-1e-5: best=vl; st={k:v.clone() for k,v in net.state_dict().items()}; pat=0
        else: pat+=1
        if pat>=10: break
    net.load_state_dict(st); return net,(mu,sd)

def predict_one(net,scaler,fr,feats,g,cap):
    mu,sd=scaler
    X=torch.tensor(((fr[feats]-mu)/sd).to_numpy(np.float32),device=DEV)
    gid=torch.full((len(fr),),g-1,dtype=torch.long,device=DEV)
    net.eval()
    with torch.no_grad(): p=net(X,gid).cpu().numpy()
    return np.clip(p,0,1)*cap

FOLDS={2023:[2022],2024:[2022,2023]}
CACHE={}
for vy,tys in FOLDS.items():
    ent={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]; yr=fr.kst_dtm.dt.year
        tr=fr[yr.isin(tys)]; va=fr[yr==vy]
        if len(tr)==0 or len(va)==0: continue
        iso=W.fit_powercurve(tr,tgt,cap)
        ent[g]=(W.with_pc(tr,iso),W.with_pc(va,iso))
    CACHE[vy]=ent
print("ready")

v7: 65 v9: 77


ready


## 1. v7 vs +temporal — 2폴드 + 편향 가드

In [2]:
def run(feats,label):
    res={}; bias={}
    for vy,ent in CACHE.items():
        pool=[]
        for g,(tr2,_) in ent.items():
            p=tr2[feats+["kst_dtm"]].copy()
            p["cf"]=tr2[TGT[g]]/W.CAP[g]; p["gid"]=g-1; p["w"]=wvec(tr2); pool.append(p)
        pool=pd.concat(pool,ignore_index=True)
        acc={g:[] for g in ent}
        for sd_ in SEEDS:
            net,scaler=train_one(pool,feats,sd_)
            for g,(_,va2) in ent.items():
                acc[g].append(predict_one(net,scaler,va2,feats,g,W.CAP[g]))
        nm=[]; fi=[]; cfs=[]
        for g,(tr2,va2) in ent.items():
            cap=W.CAP[g]; tgt=TGT[g]; w=wvec(tr2)
            lg_=lgb.LGBMRegressor(**GBM_PARAMS).fit(tr2[feats],tr2[tgt],sample_weight=w)
            hg_=HGB(loss="absolute_error",max_iter=600,learning_rate=0.03,max_leaf_nodes=63,
                l2_regularization=1.0,random_state=42).fit(tr2[feats].to_numpy(),tr2[tgt].to_numpy(),sample_weight=w)
            gp=np.clip(0.5*(lg_.predict(va2[feats])+hg_.predict(va2[feats].to_numpy())),0,cap)
            p=np.clip((1-BLEND)*gp+BLEND*np.mean(acc[g],axis=0),0,cap)
            a,b=group_scores(va2[tgt].to_numpy(),p,cap); nm.append(a); fi.append(b)
            cfs.append(100*p.mean()/cap)
        res[vy]=0.5*(1-np.mean(nm))+0.5*np.mean(fi); bias[vy]=np.mean(cfs)
    print(f"{label}: 2023={res[2023]:.4f} 2024={res[2024]:.4f} | 평균예측CF {bias[2023]:.1f}%/{bias[2024]:.1f}%")
    return res,bias

base,base_b=run(V7_FEATS,"v7 base  ")
temp,temp_b=run(V9_FEATS,"+temporal")

ok=(temp[2023]>=base[2023]) and (temp[2024]>=base[2024]) and max(temp_b.values())<=38.0
print("\n판정:", "채택 → v9 진행" if ok else "기각 → v7 유지")
json.dump(dict(adopted=bool(ok),
    base={"2023":round(base[2023],4),"2024":round(base[2024],4)},
    temporal={"2023":round(temp[2023],4),"2024":round(temp[2024],4)},
    bias_pct={"2023":round(temp_b[2023],1),"2024":round(temp_b[2024],1)}),
    open("submission/ver_8/tempfeat_result.json","w"),ensure_ascii=False,indent=2)
print("saved tempfeat_result.json")

v7 base  : 2023=0.6098 2024=0.6176 | 평균예측CF 33.5%/28.7%


+temporal: 2023=0.5980 2024=0.6131 | 평균예측CF 32.5%/28.5%

판정: 기각 → v7 유지
saved tempfeat_result.json
